In [1]:
from pynwb import NWBHDF5IO
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

In [2]:
file_path = r"C:\Users\kianb\Projects\mouse-v1-research\data\session_757216464\session_757216464.nwb"
io=NWBHDF5IO(file_path, mode="r")
nwbfile = io.read()
#print(nwbfile.session_id)
#print(nwbfile.session_description)
#print(nwbfile.session_start_time)
#print(nwbfile.institution)

#print("Total units:", len(nwbfile.units))

#print("Unit columns:", nwbfile.units.colnames)
units_df = nwbfile.units.to_dataframe()
units_df = units_df.reset_index(names="unit_id")
#print(units_df.head())


channels = pd.read_csv("../data/channels.csv")

In [3]:
#print(units_df['spike_times'])
merged_df = pd.merge(units_df, channels, left_on='peak_channel_id', right_on='id', how='left')
visp_units = merged_df[merged_df["ecephys_structure_acronym"] == "VISp"]

In [4]:
selected_units = visp_units[(visp_units['amplitude_cutoff'] < 0.1) & (visp_units['isi_violations'] < 0.5) & (visp_units['presence_ratio']>0.9) & (visp_units['quality'] == 'good')]
#print(Quality_pass['spike_times'],len(Quality_pass['spike_times']))

In [5]:
gratings = nwbfile.intervals['drifting_gratings_presentations']
gratings_df = gratings.to_dataframe()

valid_gratings = gratings_df[
    (gratings_df['orientation'].notna()) &
    (gratings_df['temporal_frequency'].notna())
]

print("Valid presentations:", len(valid_gratings))

Valid presentations: 600


In [6]:
print("Selected units:", len(selected_units))
print("Unique unit IDs:", selected_units["unit_id"].is_unique)
print(selected_units["unit_id"].head())

Selected units: 87
Unique unit IDs: True
2237    951800831
2243    951800933
2251    951801070
2257    951801206
2268    951801318
Name: unit_id, dtype: int64


In [7]:
print(valid_gratings[['start_time', 'stop_time']].head())

presentation_ids = valid_gratings.index.to_numpy()
unit_ids = selected_units['unit_id'].to_numpy()

X = np.zeros((len(valid_gratings), len(selected_units)), dtype=int)

for i, presentation in enumerate(valid_gratings.itertuples()):
    start = presentation.start_time
    stop = presentation.stop_time
    for j, neuron in enumerate(selected_units.itertuples()):
        spike_times = neuron.spike_times
        count = np.sum((spike_times >= start) & (spike_times < stop))
        X[i,j] = count

     start_time    stop_time
id                          
0   1586.113585  1588.115245
1   1589.116095  1591.117775
2   1592.118605  1594.120275
3   1595.121125  1597.122785
4   1598.123625  1600.125295


In [8]:
print("Matrix shape:", X.shape)
print("First 3 presentations, first 5 neurons:")
print(X[:3, :5])
print("Minimum spike count:", X.min())
print("Maximum spike count:", X.max())

Matrix shape: (600, 87)
First 3 presentations, first 5 neurons:
[[ 1  7 22 43  1]
 [19 50 33 21 28]
 [ 0 19 36 12  0]]
Minimum spike count: 0
Maximum spike count: 315


In [9]:
# Independently verify selected spike counts

for i, j in [(0, 0), (1, 1)]:

    presentation = valid_gratings.iloc[i]
    neuron = selected_units.iloc[j]

    start = presentation['start_time']
    stop = presentation['stop_time']
    spike_times = np.asarray(neuron['spike_times'])

    # Verify timestamps are sorted
    assert np.all(np.diff(spike_times) >= 0)

    # Find spike positions at the interval boundaries
    start_index = np.searchsorted(spike_times, start, side='left')
    stop_index = np.searchsorted(spike_times, stop, side='left')

    independent_count = stop_index - start_index

    print("Presentation:", presentation_ids[i])
    print("Neuron:", unit_ids[j])
    print("Independent count:", independent_count)
    print("Matrix count:", X[i, j])

    assert independent_count == X[i, j]

print("Verification passed!")

Presentation: 0
Neuron: 951800831
Independent count: 1
Matrix count: 1
Presentation: 1
Neuron: 951800933
Independent count: 50
Matrix count: 50
Verification passed!
